# 08 KIỂM THỬ CÁC THẾ CỜ CHIẾN THUẬT & LUẬT BIÊN (TACTICAL SCENARIO TESTING)

Notebook này trực quan hóa và kiểm thử 15 kịch bản chiến thuật chuyên biệt (tương ứng bộ test chuẩn trong `tests/test_rule_scenarios.py`):
1. **Quy tắc hình học & vật cản**: Xe tăng bị cản, tầm di chuyển và tầm ăn của Tư lệnh.
2. **Quy tắc Pháo binh**: Bị cản khi đi thường vs Bắn xuyên nhiều tầng vật cản khi ăn quân.
3. **Quy tắc cấm Tư lệnh đối mặt & Ghim quân (Pinning)**: Cấm tự sát và ghim quân cản duy nhất.
4. **Quy tắc vượt sông & Đoạn ngầm**: Sông sâu vs Ngầm lội đá (Ford), ngoại lệ ăn quân thế chỗ qua sông.
5. **Tương tác bờ biển & Hải quân**: Pháo đất liền bắn Tàu bằng BOMBARD, Hải quân không lên đất liền.
6. **Quy tắc phong Anh hùng**: Tự động phong Anh hùng khi còn 1 quân bảo vệ, sức mạnh Xe tăng Anh hùng.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Cấu hình nạp module
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "packages").is_dir() and (p / "research").is_dir()), None)
if str(PROJECT_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "packages"))

from commander_game_engine.board import Board, EdgeType, Position, Terrain
from commander_game_engine.game_state import GameState, Piece, PieceType, Player
from commander_game_engine.actions import ActionType, AtomicAction
from commander_game_engine.move_generator import generate_piece_actions
from commander_game_engine.engine import apply_action
from commander_game_engine import rules

print("Nạp module Game Engine phục vụ kiểm thử kịch bản thành công!")

Nạp module Game Engine phục vụ kiểm thử kịch bản thành công!


## Nhóm 1: Hình học di chuyển & Vật cản (Xe tăng & Tư lệnh)

In [2]:
board = Board()

# Kịch bản 1: Xe tăng đi 4 hướng 2 đoạn, bị cản nếu đoạn 1 có quân
st1 = GameState(board=board, current_player=Player.RED)
tank = Piece(id="red_tank", piece_type=PieceType.TANK, owner=Player.RED, position=(3, 5))
st1.add_piece(tank)

acts_open = generate_piece_actions(st1, "red_tank")
dest_open = {a.end for a in acts_open}
assert dest_open == {(1, 5), (2, 5), (4, 5), (5, 5), (3, 3), (3, 4), (3, 6), (3, 7)}
print(f"Kịch bản 1.1: Xe tăng giữa đất trống -> Đúng 8 nước đi thẳng 1-2 đoạn: {dest_open}")

# Thêm quân cản tại (3, 6)
st1.add_piece(Piece(id="blocker", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(3, 6)))
acts_block = generate_piece_actions(st1, "red_tank")
dest_block = {a.end for a in acts_block}
assert (3, 6) not in dest_block and (3, 7) not in dest_block
print(f"Kịch bản 1.2: Có quân cản tại (3, 6) -> Mất cả (3, 6) và (3, 7). PASS!")

# Kịch bản 2: Tư lệnh đi xa không giới hạn, nhưng chỉ ăn quân cự ly 1 đoạn
st2 = GameState(board=board, current_player=Player.RED)
cmd = Piece(id="red_cmd", piece_type=PieceType.COMMANDER, owner=Player.RED, position=(0, 6))
enemy_near = Piece(id="blue_near", piece_type=PieceType.INFANTRY, owner=Player.BLUE, position=(1, 6))
enemy_far = Piece(id="blue_far", piece_type=PieceType.INFANTRY, owner=Player.BLUE, position=(0, 8))
st2.add_piece(cmd)
st2.add_piece(enemy_near)
st2.add_piece(enemy_far)

acts_cmd = generate_piece_actions(st2, "red_cmd")
cap_cmd = {a.end for a in acts_cmd if a.action_type == ActionType.CAPTURE}
assert (1, 6) in cap_cmd
assert (0, 8) not in cap_cmd
print(f"Kịch bản 2: Tư lệnh ăn quân cự ly 1 (1, 6) thành công; không được ăn cự ly 2 (0, 8). PASS!")

Kịch bản 1.1: Xe tăng giữa đất trống -> Đúng 8 nước đi thẳng 1-2 đoạn: {(5, 5), (3, 4), (1, 5), (3, 7), (4, 5), (3, 3), (3, 6), (2, 5)}
Kịch bản 1.2: Có quân cản tại (3, 6) -> Mất cả (3, 6) và (3, 7). PASS!
Kịch bản 2: Tư lệnh ăn quân cự ly 1 (1, 6) thành công; không được ăn cự ly 2 (0, 8). PASS!


## Nhóm 2: Đặc tính Pháo binh (Bắn xuyên qua vật cản)

In [3]:
board = Board()

# Kịch bản 3: Pháo binh đi thường bị cản bởi quân đứng giữa
st3 = GameState(board=board, current_player=Player.RED)
art = Piece(id="red_art", piece_type=PieceType.ARTILLERY, owner=Player.RED, position=(2, 5))
obs = Piece(id="obs", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(2, 6))
st3.add_piece(art)
st3.add_piece(obs)

acts_art = generate_piece_actions(st3, "red_art")
moves_art = {a.end for a in acts_art if a.action_type == ActionType.MOVE}
assert (2, 7) not in moves_art and (2, 8) not in moves_art
print("Kịch bản 3: Pháo đi thường không được xuyên qua quân cản (2, 6). PASS!")

# Kịch bản 4: Pháo binh bắn xuyên qua 2 quân cản liên tiếp để tiêu diệt mục tiêu
st4 = GameState(board=board, current_player=Player.RED)
st4.add_piece(art)
st4.add_piece(Piece(id="obs1", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(2, 6)))
st4.add_piece(Piece(id="obs2", piece_type=PieceType.TANK, owner=Player.BLUE, position=(2, 7)))
st4.add_piece(Piece(id="target", piece_type=PieceType.INFANTRY, owner=Player.BLUE, position=(2, 8)))

acts_art_4 = generate_piece_actions(st4, "red_art")
attacks = {a.end for a in acts_art_4 if a.action_type in (ActionType.CAPTURE, ActionType.BOMBARD)}
assert (2, 8) in attacks
print("Kịch bản 4: Pháo binh bắn hạ mục tiêu (2, 8) xuyên qua 2 quân cản liên tiếp. PASS!")

Kịch bản 3: Pháo đi thường không được xuyên qua quân cản (2, 6). PASS!
Kịch bản 4: Pháo binh bắn hạ mục tiêu (2, 8) xuyên qua 2 quân cản liên tiếp. PASS!


## Nhóm 3: Cấm lộ mặt Tư lệnh & Cơ chế ghim quân (Pinning)

In [4]:
board = Board()

# Kịch bản 5: Tư lệnh không được tự di chuyển vào thế đối mặt
st5 = GameState(board=board, current_player=Player.RED)
st5.add_piece(Piece(id="red_cmd", piece_type=PieceType.COMMANDER, owner=Player.RED, position=(0, 5)))
st5.add_piece(Piece(id="blue_cmd", piece_type=PieceType.COMMANDER, owner=Player.BLUE, position=(11, 6)))

acts_cmd_5 = generate_piece_actions(st5, "red_cmd")
assert (0, 6) not in {a.end for a in acts_cmd_5}
print("Kịch bản 5: Tư lệnh tự đi vào cột 6 trống đối mặt Tướng địch -> Bị loại bỏ. PASS!")

# Kịch bản 6: Ghim quân cản duy nhất (Bộ binh ở giữa không được rời cột)
st6 = GameState(board=board, current_player=Player.RED)
st6.add_piece(Piece(id="red_cmd", piece_type=PieceType.COMMANDER, owner=Player.RED, position=(0, 6)))
st6.add_piece(Piece(id="blue_cmd", piece_type=PieceType.COMMANDER, owner=Player.BLUE, position=(11, 6)))
st6.add_piece(Piece(id="red_inf", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(4, 6)))

acts_pin = generate_piece_actions(st6, "red_inf")
dests_pin = {a.end for a in acts_pin}
assert (4, 5) not in dests_pin and (4, 7) not in dests_pin  # Không được đi ngang
assert (3, 6) in dests_pin and (5, 6) in dests_pin          # Được đi dọc duy trì che chắn
print(f"Kịch bản 6: Quân cản duy nhất bị ghim chặt, chỉ được đi dọc: {dests_pin}. PASS!")

Kịch bản 5: Tư lệnh tự đi vào cột 6 trống đối mặt Tướng địch -> Bị loại bỏ. PASS!
Kịch bản 6: Quân cản duy nhất bị ghim chặt, chỉ được đi dọc: {(5, 6), (3, 6)}. PASS!


## Nhóm 4: Vượt sông sâu, Ngầm lội đá & Ngoại lệ ăn quân

In [5]:
board = Board()

# Kịch bản 7: Quân nặng không được đi thường qua sông sâu
st7 = GameState(board=board, current_player=Player.RED)
st7.add_piece(Piece(id="red_art", piece_type=PieceType.ARTILLERY, owner=Player.RED, position=(5, 3)))
assert board.get_edge_type((5, 3), (6, 3)) == EdgeType.DEEP_RIVER
acts_river = generate_piece_actions(st7, "red_art")
assert (6, 3) not in {a.end for a in acts_river}
print("Kịch bản 7: Pháo binh đi thường qua nước sâu (5, 3)->(6, 3) bị chặn. PASS!")

# Kịch bản 8: Ngoại lệ Q02 - Quân nặng được ăn quân thế chỗ qua sông sâu
st8 = GameState(board=board, current_player=Player.RED)
st8.add_piece(Piece(id="red_art", piece_type=PieceType.ARTILLERY, owner=Player.RED, position=(5, 3)))
st8.add_piece(Piece(id="blue_inf", piece_type=PieceType.INFANTRY, owner=Player.BLUE, position=(6, 3)))
acts_cap_river = generate_piece_actions(st8, "red_art")
assert (6, 3) in {a.end for a in acts_cap_river if a.action_type == ActionType.CAPTURE}
print("Kịch bản 8: Pháo binh ăn quân thế chỗ qua sông sâu (5, 3)->(6, 3) thành công. PASS!")

# Kịch bản 9: Mọi quân qua được ngầm (x=5)
st9 = GameState(board=board, current_player=Player.RED)
st9.add_piece(Piece(id="red_art", piece_type=PieceType.ARTILLERY, owner=Player.RED, position=(5, 5)))
assert board.get_edge_type((5, 5), (6, 5)) == EdgeType.FORD
acts_ford = generate_piece_actions(st9, "red_art")
assert (6, 5) in {a.end for a in acts_ford}
print("Kịch bản 9: Pháo binh đi qua ngầm đá (5, 5)->(6, 5) thành công. PASS!")

# Kịch bản 10: Hải quân không được qua ngầm
st10 = GameState(board=board, current_player=Player.BLUE)
st10.add_piece(Piece(id="blue_navy", piece_type=PieceType.NAVY, owner=Player.BLUE, position=(6, 5)))
acts_navy_ford = generate_piece_actions(st10, "blue_navy")
assert (5, 5) not in {a.end for a in acts_navy_ford}
print("Kịch bản 10: Hải quân qua đoạn ngầm bị chặn nghiêm ngặt. PASS!")

Kịch bản 7: Pháo binh đi thường qua nước sâu (5, 3)->(6, 3) bị chặn. PASS!
Kịch bản 8: Pháo binh ăn quân thế chỗ qua sông sâu (5, 3)->(6, 3) thành công. PASS!
Kịch bản 9: Pháo binh đi qua ngầm đá (5, 5)->(6, 5) thành công. PASS!
Kịch bản 10: Hải quân qua đoạn ngầm bị chặn nghiêm ngặt. PASS!


## Nhóm 5: Tác chiến Hải quân & Bờ biển

In [6]:
board = Board()

# Kịch bản 11: Pháo đất liền bắn Tàu trên biển bằng BOMBARD
st11 = GameState(board=board, current_player=Player.RED)
st11.add_piece(Piece(id="red_art", piece_type=PieceType.ARTILLERY, owner=Player.RED, position=(3, 3)))
st11.add_piece(Piece(id="blue_navy", piece_type=PieceType.NAVY, owner=Player.BLUE, position=(3, 1)))

acts_bomb = generate_piece_actions(st11, "red_art")
navy_attacks = [a for a in acts_bomb if a.end == (3, 1)]
assert len(navy_attacks) == 1
assert navy_attacks[0].action_type == ActionType.BOMBARD
res_bomb = apply_action(st11, navy_attacks[0])
assert res_bomb.state.pieces["red_art"].position == (3, 3) # Giữ nguyên vị trí
assert "blue_navy" not in res_bomb.state.pieces
print("Kịch bản 11: Pháo đất liền bắn Tàu bằng BOMBARD thành công, pháo không nhảy xuống biển. PASS!")

# Kịch bản 12: Hải quân không được lên đất liền
st12 = GameState(board=board, current_player=Player.BLUE)
st12.add_piece(Piece(id="blue_navy", piece_type=PieceType.NAVY, owner=Player.BLUE, position=(4, 2)))
acts_navy_land = generate_piece_actions(st12, "blue_navy")
assert (4, 3) not in {a.end for a in acts_navy_land}
print("Kịch bản 12: Hải quân không thể di chuyển vào đất liền (4, 3). PASS!")

Kịch bản 11: Pháo đất liền bắn Tàu bằng BOMBARD thành công, pháo không nhảy xuống biển. PASS!
Kịch bản 12: Hải quân không thể di chuyển vào đất liền (4, 3). PASS!


## Nhóm 6: Tự động phong Anh hùng & Nâng cấp sức mạnh

In [7]:
board = Board()

# Kịch bản 13: Tự động phong Anh hùng khi chỉ còn 1 quân bảo vệ Tư lệnh
st13 = GameState(board=board, current_player=Player.RED)
st13.add_piece(Piece(id="red_cmd", piece_type=PieceType.COMMANDER, owner=Player.RED, position=(0, 6)))
st13.add_piece(Piece(id="red_tank", piece_type=PieceType.TANK, owner=Player.RED, position=(3, 5)))
# Phe Xanh còn nhiều quân
st13.add_piece(Piece(id="blue_cmd", piece_type=PieceType.COMMANDER, owner=Player.BLUE, position=(11, 6)))
st13.add_piece(Piece(id="blue_inf", piece_type=PieceType.INFANTRY, owner=Player.BLUE, position=(10, 2)))

res_hero = apply_action(st13, AtomicAction(ActionType.MOVE, "red_tank", (3, 5), (3, 6)))
assert res_hero.accepted == True
assert res_hero.state.pieces["red_tank"].is_heroic == True
print("Kịch bản 13: Xe tăng tự động được phong Anh hùng khi là quân cuối cùng bảo vệ Tư lệnh. PASS!")

# Kịch bản 14: Sức mạnh Xe tăng Anh hùng (8 hướng x 3 đoạn = 24 nước đi)
st14 = GameState(board=board, current_player=Player.RED)
st14.add_piece(Piece(id="hero_tank", piece_type=PieceType.TANK, owner=Player.RED, position=(5, 5), is_heroic=True))
acts_hero = generate_piece_actions(st14, "hero_tank")
dests_hero = {a.end for a in acts_hero}
assert len(dests_hero) == 24
assert (8, 8) in dests_hero and (2, 2) in dests_hero
print(f"Kịch bản 14: Xe tăng Anh hùng có đầy đủ 8 hướng tầm 3 ({len(dests_hero)} nước đi). PASS!")

Kịch bản 13: Xe tăng tự động được phong Anh hùng khi là quân cuối cùng bảo vệ Tư lệnh. PASS!
Kịch bản 14: Xe tăng Anh hùng có đầy đủ 8 hướng tầm 3 (24 nước đi). PASS!


## 7. Bảng tổng kết 15 kịch bản chiến thuật
Toàn bộ 15 kịch bản chiến thuật biên đặc thù của Cờ Tư Lệnh đã được kiểm tra độc lập và vượt qua 100% với độ tin cậy tuyệt đối.